## 💳 Credit Card Fraud Detection

### 📌 Problem Statement

The objective of this project is to build a Machine Learning model that can accurately detect fraudulent credit card transactions.

Credit card fraud is a critical issue in the financial industry, where fraudulent transactions are extremely rare compared to legitimate ones. This creates a **highly imbalanced dataset**, making the problem more challenging.

The model should be able to:
- Identify fraudulent transactions (Class = 1)
- Distinguish them from legitimate transactions (Class = 0)

In [ ]:
# Data manipulation
import numpy as np
import pandas as pd

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Handling imbalanced data
from imblearn.over_sampling import SMOTE

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv("../Data/creditcard.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
# Check duplicate rows
duplicates = df.duplicated().sum()
print("Number of duplicate rows:", duplicates)

# Remove duplicates
df = df.drop_duplicates()

# Verify removal
print("Shape after removing duplicates:", df.shape)

In [ ]:
df["Class"].value_counts()

## 🧹 Data Cleaning

In this step, we clean the dataset to improve data quality.

## 🎯 Objectives

- Check for duplicate records
- Remove duplicate entries (if any)

In [ ]:
# Check duplicate rows
duplicates = df.duplicated().sum()
print("Number of duplicate rows:", duplicates)

# Remove duplicates
df = df.drop_duplicates()

# Verify removal
print("Shape after removing duplicates:", df.shape)

### 🚫 Handling Missing Values

In this dataset, we checked for missing values and found that:

- There are **no missing values** in any column

#### ✅ Conclusion

- No imputation or treatment is required
- Dataset is clean and ready for further processing

In [ ]:
# Already verified in previous step
df.isnull().sum()

## 📊 Exploratory Data Analysis (EDA)

EDA helps us understand the dataset visually and statistically before model training.

Class Distribution Analysis

In fraud detection, understanding class distribution is extremely important because the dataset is highly imbalanced.


In [ ]:
# Class distribution count
print(df['Class'].value_counts())

# Visualization
plt.figure(figsize=(6,4))

sns.countplot(x='Class', data=df)

plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")

plt.show()

In [ ]:
# Percentage distribution
class_percent = df['Class'].value_counts(normalize=True) * 100

print(class_percent)

### 💰 Transaction Amount Distribution

We analyze how transaction amounts are distributed.

- Range of transactions
- Presence of outliers
- Skewness in data

In [ ]:
plt.figure(figsize=(10,5))

sns.histplot(df['Amount'], bins=50, kde=True)

plt.title("Transaction Amount Distribution")
plt.xlabel("Amount")
plt.ylabel("Frequency")

plt.show()

### 🔍 Fraud vs Normal Transaction Amount

We compare transaction amounts between:
- Legitimate transactions
- Fraudulent transactions

This helps identify behavioral differences.

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(x='Class', y='Amount', data=df)

plt.title("Fraud vs Normal Transaction Amount")
plt.show()

### 🌡️ Correlation Heatmap

A correlation heatmap helps us understand relationships between numerical features.

- Positive correlation → variables increase together
- Negative correlation → one increases while the other decreases

In [ ]:
plt.figure(figsize=(20,15))

sns.heatmap(df.corr(), cmap='coolwarm')

plt.title("Correlation Heatmap")

plt.show()

### 🎯 Feature and Target Split

Before training Machine Learning models, we separate:

- Features (Independent Variables)
- Target (Dependent Variable)

#### 📌 Features (X)

Input variables used to predict fraud:
- V1 to V28
- Time
- Amount

#### 🎯 Target (y)

`Class` column:
- 0 → Normal transaction
- 1 → Fraud transaction

In [ ]:
# Features
X = df.drop('Class', axis=1)

# Target
y = df['Class']

# Display shapes
print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Display shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

### 📏 Feature Scaling

Feature scaling standardizes numerical values so that all features contribute equally during model training.

#### 🎯 Why Scaling is Important?

Some features may have very large values while others have small values.

Example:
- `Amount` → can be very large
- PCA features (`V1-V28`) → already scaled around similar ranges

Without scaling:
- Models may become biased toward larger-value features
- Distance-based algorithms may perform poorly

#### ✅ Technique Used

We use:
- `StandardScaler`

It transforms data into:
- Mean = 0
- Standard Deviation = 1

In [ ]:
# Initialize scaler
scaler = StandardScaler()

# Scale training and testing data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Display shapes
print("Scaled X_train shape:", X_train_scaled.shape)
print("Scaled X_test shape:", X_test_scaled.shape)

### ⚖️ Handling Imbalanced Data using SMOTE

The dataset is highly imbalanced:
- Majority Class → Normal Transactions (0)
- Minority Class → Fraud Transactions (1)

Machine Learning models may become biased toward the majority class.

#### 🎯 Objective

Balance the training dataset so the model can learn fraud patterns properly.

#### ✅ Technique Used

SMOTE (Synthetic Minority Oversampling Technique)

SMOTE creates synthetic samples of the minority class instead of simply duplicating rows.

In [ ]:
# Initialize SMOTE
smote = SMOTE(random_state=42)

# Apply SMOTE only on training data
X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

# Check class distribution after SMOTE
print("Before SMOTE:\n")
print(y_train.value_counts())

print("\nAfter SMOTE:\n")
print(pd.Series(y_train_smote).value_counts())

### 📈 Logistic Regression

Logistic Regression is a popular classification algorithm used for binary classification problems.

It predicts probabilities and classifies data into:
- 0 → Normal Transaction
- 1 → Fraud Transaction

In [ ]:
# Initialize model
lr_model = LogisticRegression()

# Train model
lr_model.fit(X_train_smote, y_train_smote)

# Predictions
y_pred_lr = lr_model.predict(X_test_scaled)

print("Logistic Regression model trained successfully.")

### 🌳 Decision Tree Classifier

Decision Trees split data into branches based on feature conditions.

They are:
- Easy to understand
- Powerful for nonlinear patterns
- Able to capture complex relationships

In [ ]:
# Initialize model
dt_model = DecisionTreeClassifier(random_state=42)

# Train model
dt_model.fit(X_train_smote, y_train_smote)

# Predictions
y_pred_dt = dt_model.predict(X_test_scaled)

print("Decision Tree model trained successfully.")

### 🌲 Random Forest Classifier

Random Forest is an ensemble learning algorithm.

It combines multiple Decision Trees and makes predictions based on majority voting.

#### ✅ Advantages

- More stable than a single tree
- Better generalization
- Reduces overfitting

In [ ]:
# Initialize model
rf_model = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

# Train model
rf_model.fit(X_train_smote, y_train_smote)

# Predictions
y_pred_rf = rf_model.predict(X_test_scaled)

print("Random Forest model trained successfully.")

### 📊 Model Evaluation

In this step, we evaluate all trained models using classification metrics.

#### 🎯 Objective

Measure how well the models detect fraudulent transactions.

#### 📌 Evaluation Metrics

1. Accuracy
2. Precision
3. Recall
4. F1-Score

#### ⚠️ Important Note

For imbalanced datasets like fraud detection:
- Accuracy alone is NOT reliable
- Recall and F1-score are more important

In [ ]:
# Create comparison table

model_comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest'],
    
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf)
    ],
    
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_rf)
    ],
    
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_rf)
    ],
    
    'F1-Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_rf)
    ]
})

# Display comparison table
model_comparison

In [ ]:
# Confusion matrix for Random Forest

cm = confusion_matrix(y_test, y_pred_rf)

# Plot heatmap
plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Random Forest Confusion Matrix")

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.show()

### 💾 Save Best Model

After evaluating all models, Random Forest performed the best.

Now we save the trained model so it can be reused later without retraining.

In [32]:
# Load saved model
import pickle

# Save Random Forest model
with open("../Model/fraud_detection_model.pkl", "wb") as file:
    pickle.dump(rf_model, file)

print("Model saved successfully.")

Model saved successfully.
